**STEP 1**

The goal of this 12-step program is to show you an enhanced library for string pattern matching, using the Python programming language.

First, let's review regular expressions that are built in the Python programming langauge. The module is named "re", also refered as RE.

Say we are using regular expressions to analyze a text that contains some unknown program, written in a mystery programming language. Say the following is a line of text in mystery code:

* DEFINE('f(x,y,z)c,i,n,s,t,v')

You are told that this line of code invokes the DEFINE function. It declares a new function based on what is written between the single quotes. In the example above, it is declaring a function named f that has three parameters named x, y, z, and six local variables named c, i, n, s, t, and v.

Let's write Python regular expressions that extracts all of these names (the name of the function, the names of the parameters and the names of the local variables). This program will store this information in three variables and if we are successful, this will be the result:

* function = 'f',
* parameters = ['x', 'y', 'z'],
* variables = ['c', 'i', 'n', 's', 't', 'v'].

In [1]:
        subject = "DEFINE('f(x,y,z)c,i,n,s,t,v')"
        lc = &LCASE

**STEP 2**

Let's try using the *re.search* function. There are two well known elements for regular expressions:

*   the '^' (*caret*) character to specify the pattern ensuring the beginning of the text, also referred to as left-position zero, and
*   the '$' (*dollar-sign*) character to specify the pattern ensuring the end of the text, also referred to as right-position zero.

After a successful regular expression match, the *re.search* returns an *re.Match* object. To nicely display any Python object, the pretty-print function, *pprint*, is used. To gain access to any captured groups being returned, the *re.Match.groups* and *re.Match.groupdict* are used. The *groups* function returns a list of positionally captured groups and the *groupdict* function returns a dictionary of named captured groups.\
\
This is going to be amazing! You mean all I have to do is create just one regular expression pattern to verify the validity of the text and simulataneously extract all the separate elements into structured data in *one fell swoop*?

In [2]:
        pat = POS(0) "DEFINE('" ANY(lc) . function "(" ANY(lc) . p1
+             ARBNO("," ANY(lc) . p) ")" ANY(lc) . v1
+             ARBNO("," ANY(lc) . v) "')" RPOS(0)
        subject pat                                             :F(no3)
        OUTPUT = "Matched: " function " " p1 " " p " " v1 " " v :(end3)
no3     OUTPUT = "Unmatched!"
end3

Matched: f x z c v


**STEP 3**

What happened!? I can't do my work.\
Where is the y parameter name?\
Where are most of the variable names?\
Where are the two lists for the repetions?\
I think I might have choosen unwisely.\
\
Can it be that the construct (?:___) specifying a non-capturing group is in some way interferring with properly capturing its nested pattern? So let's modify these to simply be capturing groups instead.

In [3]:
        pat = POS(0) "DEFINE('" ANY(lc) . function "(" ANY(lc) . p1
+             ARBNO(("," ANY(lc) . p) . pg) ")" ANY(lc) . v1
+             ARBNO(("," ANY(lc) . v) . vg) "')" RPOS(0)
        subject pat                                             :F(no5)
        OUTPUT = "Matched: " function " " p1 " " pg " " p " " v1 " " vg " " v :(end5)
no5     OUTPUT = "Unmatched!"
end5

Matched: f x ,z z c ,v v


**STEP 4**

What happened! I still can't do my work.\
If anything, things seem worse than before.\
I think I might have choosen unwisely.\
\
Can it be that capturing elements of a reptition just isn't possible?\
Can it be that the *re.search* function will just not return a list of elements?\
Unfortunately, the RE module is working as designed and hence this limitation will likely never to be lifted. It will only ever just return the last element of the repitition, and will always refuse to capture the remaining parts of the repitition.\
\
Unfortunately, the developers of the RE module might just have built this bug into the product as a feature. So, let's try one more time. Let's attempt to capture the entire text of these lists and not capture their nested elements.

In [4]:
        pat = POS(0) "DEFINE('" ANY(lc) . function "(" ANY(lc) . p1
+             ARBNO("," ANY(lc)) . prest ")" ANY(lc) . v1
+             ARBNO("," ANY(lc)) . vrest "')" RPOS(0)
        subject pat                                             :F(no7)
        OUTPUT = "Matched: " function " " p1 " " prest " " v1 " " vrest :(end7)
no7     OUTPUT = "Unmatched!"
end7

Matched: f x ,y,z c ,i,n,s,t,v


**STEP 5**

What happened? I'll never get my work done.\
\
So, it appears this caliber of results is the best we can accomplish using the RE module, and that our initial desire to develop one regular expression pattern to extract structured data from text will not be fulfilled. For this task, it appears more coding will be necessary. Oh but I desperately wanted to avoid procedural coding all together. I wanted the pattern to look similar to the subject. I want the solution to resemble the problem.\
\
Given that we are limited to just a single capture group returning a repitition in its entirety, and since individual elements can not be captured, let's merge the patterns which return the first and remaining parts into one pattern for a proper comma-seperated string for later processing.

In [5]:
        names = ANY(lc) ARBNO("," ANY(lc))
        pat = POS(0) "DEFINE('" ANY(lc) . function "(" names . params ")"
+             names . vars "')" RPOS(0)
        subject pat                                             :F(no9)
        OUTPUT = "Matched: " function " " params " " vars       :(end9)
no9     OUTPUT = "Unmatched!"
end9

Matched: f x,y,z c,i,n,s,t,v


**STEP 6**

So what happens now? I've got so much work and not enough time.\
I must write more Python code! Ooh, somebody stop me.\
If the elements of a list, referred below as items, are processed in Python code seperately and revalidating these items would be necessary, the following code is representative of what's necessary.

In [6]:
        subject pat                                             :F(no11)
        parameters =
        rest = params
par11   rest ("," | "") ANY(lc) . item =                        :F(pard11)
        parameters = parameters " " item                        :(par11)
pard11  variables =
        rest = vars
var11   rest ("," | "") ANY(lc) . item =                        :F(vard11)
        variables = variables " " item                          :(var11)
vard11  OUTPUT = function " [" parameters " ] [" variables " ]" :(end11)
no11    OUTPUT = "Unmatched!"
end11

f [ x y z ] [ c i n s t v ]


**STEP 7**

So what happened there? I created a beautiful mess.\
\
But it works. It validates the input text and produces three variables, one containing the function name, and two containing the list of names for parameters and variables.\
\
Also, if revalidation is not necessary, this code can be simplified further.

In [7]:
        subject pat                                             :F(no13)
        parameters = REPLACE(params, ",", " ")
        variables = REPLACE(vars, ",", " ")
        OUTPUT = "Matched: " function " [ " parameters " ] [ " variables " ]" :(end13)
no13    OUTPUT = "Unmatched!"
end13

Matched: f [ x y z ] [ c i n s t v ]


**STEP 8**

Can there be another way?\
Let's try using the SNOBOL4python library as an alternative.\
The following code will mount and import the SNOBOL4python library.

In [8]:
*       Nothing to install and nothing to import: patterns are part of the language.
*       One helper: push(name) appends the current item to the list variable called name.
        DEFINE('push(name)')                                    :(push_end)
push    $name = $name " " item                                  :(RETURN)
push_end

**STEP 9**

To use the new PATTERN datatype provided by the SNOBOL4python module:

*   r"^" becomes *POS*(0)
*   r"$" becomes *RPOS*(0)
*   r"[a-z]" becomes *ANY*(LCASE)
*   r"xyz" becomes σ('xyz'), or alternatively
*   r"xyz" becomes σ('x') + σ('y') + σ('z')
*   r"(\_\_\_)*" becomes *ARBNO*(___)
*   re.search(pattern, subject) becomes subject in PATTERN

Let's start by just getting the PATTERN to work, and for now not deal with capturing any results.

In [9]:
        subject POS(0) "DEFINE('" ANY(lc) "(" ANY(lc) ARBNO("," ANY(lc)) ")"
+               ANY(lc) ARBNO("," ANY(lc)) "')" RPOS(0)         :F(no17)
        OUTPUT = "Matched."                                     :(end17)
no17    OUTPUT = "Unmatched!"
end17

Matched.


**STEP 10**

What just happened? It matched! Is there any hope I can complete my work?

Now, let's decorate the above pattern with Python code to capture the PATTERN matching results into variables containing strings and lists.

* r"(?P<name>\_\_\_) becomes ___ % "name"\
* r"<no-can-do>" becomes λ(python_code_string)

In [10]:
        parameters =
        variables =
        subject POS(0) "DEFINE('" ANY(lc) . function "("
+               ANY(lc) $ item *push('parameters')
+               ARBNO("," ANY(lc) $ item *push('parameters')) ")"
+               ANY(lc) $ item *push('variables')
+               ARBNO("," ANY(lc) $ item *push('variables')) "')" RPOS(0) :F(no19)
        OUTPUT = "Matched. " function " [" parameters " ] [" variables " ]" :(end19)
no19    OUTPUT = "Unmatched!"
end19

Matched. f [ x y z ] [ c i n s t v ]


**STEP 11**

What happened? My work is done. It's a miracle.\
The solution does seem to resemble the problem.\
Can it really be that easy?

Now introducing the PATTERN phi, φ(r'___'). It will match a regular expression. And now a solution using regular expression patterns as an integral part of the new PATTERN datatype.

In [11]:
        parameters =
        variables =
        name  = ANY(lc)
        plist = name $ item *push('parameters') ARBNO("," name $ item *push('parameters'))
        vlist = name $ item *push('variables') ARBNO("," name $ item *push('variables'))
        subject POS(0) "DEFINE('" name . function "(" plist ")" vlist "')" RPOS(0) :F(no21)
        OUTPUT = "Matched: " function " [" parameters " ] [" variables " ]" :(end21)
no21    OUTPUT = "Unmatched!"
end21

Matched: f [ x y z ] [ c i n s t v ]


**STEP 12**

What happens next! You can whatever work you want!\
This SNOBOL4python Python module can process all four levels of the Chomsky heirarchy.\
This concludes this 12-step program.